# Neural Network Test Run (4)

In [1]:
%%capture
# %%capture prevents this cell from printing a ton of STDERR stuff to the screen

## First, check to see if lightning is installed, if not, install it.
##
## NOTE: If you **do** need to install something, just know that you may need to
##       restart your session for python to find the new module(s).
##
##       To restart your session:
##       - In Google Colab, click on the "Runtime" menu and select
##         "Restart Session" from the pulldown menu
##       - In a local jupyter notebook, click on the "Kernel" menu and select
##         "Restart Kernel" from the pulldown menu
import pip
try:
  __import__("lightning")
except ImportError:
  pip.main(['install', "lightning"])

In [2]:
import torch
import torch.nn as nn 
import torch.nn.functional as F 
from torch.optim import Adam 

import lightning as L 
from torch.utils.data import TensorDataset, DataLoader 

import pandas as pd 
from sklearn.model_selection import train_test_split

In [4]:
url = '../iris.txt'
df = pd.read_table(url, sep=",", header=None)

In [5]:
df.head()

,0,1,2,3,4
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


In [6]:
df.columns = ["sepal_length", 
              "sepal_width", 
              "petal_length", 
              "petal_width", 
              "class"]
df.head()

,sepal_length,sepal_width,petal_length,petal_width,class
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


In [7]:
df.shape

(150, 5)

In [8]:
df['class'].nunique()

3

In [9]:
df['class'].unique()

<StringArray>
['Iris-setosa', 'Iris-versicolor', 'Iris-virginica']
Length: 3, dtype: str

In [10]:
for class_name in df['class'].unique():
  print(class_name, ':', sum(df['class'] == class_name), sep="")

Iris-setosa:50
Iris-versicolor:50
Iris-virginica:50


In [11]:
df[['petal_width', 'sepal_width']].head()

,petal_width,sepal_width
0,0.2,3.5
1,0.2,3.0
2,0.2,3.2
3,0.2,3.1
4,0.2,3.6


In [12]:
input_values = df[['petal_width', 'sepal_width']]
input_values.head()

,petal_width,sepal_width
0,0.2,3.5
1,0.2,3.0
2,0.2,3.2
3,0.2,3.1
4,0.2,3.6


In [13]:
label_values = df['class']
label_values.head()

0    Iris-setosa
1    Iris-setosa
2    Iris-setosa
3    Iris-setosa
4    Iris-setosa
Name: class, dtype: str

In [14]:
classes_as_numbers = label_values.factorize()[0]
classes_as_numbers

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2])

In [15]:
input_train, input_test, label_train, label_test = train_test_split(input_values, 
                                                                    classes_as_numbers, 
                                                                    test_size=0.25, 
                                                                    stratify=classes_as_numbers, 
                                                                    random_state=42)

In [16]:
input_train.shape

(112, 2)

In [17]:
label_train.shape

(112,)

In [19]:
input_test.shape

(38, 2)

In [18]:
label_test.shape

(38,)

In [20]:
one_hot_label_train = F.one_hot(torch.tensor(label_train)).type(torch.float32)

In [21]:
label_train

array([2, 2, 1, 1, 1, 2, 0, 2, 0, 2, 0, 2, 1, 0, 0, 1, 2, 0, 0, 1, 1, 1,
       0, 1, 2, 0, 2, 1, 2, 0, 0, 1, 0, 2, 0, 0, 1, 0, 1, 0, 0, 1, 2, 2,
       0, 2, 1, 0, 2, 0, 2, 2, 0, 1, 2, 2, 1, 1, 0, 1, 1, 2, 1, 2, 0, 1,
       0, 2, 1, 2, 1, 2, 2, 0, 2, 1, 0, 2, 0, 2, 1, 1, 0, 2, 2, 0, 0, 2,
       2, 1, 2, 0, 2, 1, 2, 2, 0, 1, 1, 1, 1, 1, 0, 2, 1, 1, 0, 0, 0, 0,
       1, 0])

In [23]:
one_hot_label_train[:10]

tensor([[0., 0., 1.],
        [0., 0., 1.],
        [0., 1., 0.],
        [0., 1., 0.],
        [0., 1., 0.],
        [0., 0., 1.],
        [1., 0., 0.],
        [0., 0., 1.],
        [1., 0., 0.],
        [0., 0., 1.]])

In [24]:
max_vals_in_input_train = input_train.max()
max_vals_in_input_train

petal_width    2.5
sepal_width    4.4
dtype: float64

In [26]:
min_vals_in_input_train = input_train.min()
min_vals_in_input_train

petal_width    0.1
sepal_width    2.0
dtype: float64

In [27]:
input_train.head()

,petal_width,sepal_width
130,1.9,2.8
122,2.0,2.8
81,1.0,2.4
71,1.3,2.8
89,1.3,2.5


In [28]:
# Normalize with max and min.
input_train = (input_train - min_vals_in_input_train) / (max_vals_in_input_train - min_vals_in_input_train)
input_train.head()

,petal_width,sepal_width
130,0.750000,0.333333
122,0.791667,0.333333
81,0.375000,0.166667
71,0.500000,0.333333
89,0.500000,0.208333


In [29]:
# Normalize test
input_test = (input_test - min_vals_in_input_train) / (max_vals_in_input_train - min_vals_in_input_train)
input_test.head()

,petal_width,sepal_width
42,0.041667,0.500000
56,0.625000,0.541667
99,0.500000,0.333333
53,0.500000,0.125000
38,0.041667,0.416667


In [30]:
input_train.values

array([[0.75      , 0.33333333],
       [0.79166667, 0.33333333],
       [0.375     , 0.16666667],
       [0.5       , 0.33333333],
       [0.5       , 0.20833333],
       [1.        , 0.54166667],
       [0.08333333, 0.58333333],
       [0.70833333, 0.29166667],
       [0.04166667, 0.45833333],
       [0.70833333, 0.33333333],
       [0.125     , 0.75      ],
       [0.75      , 0.29166667],
       [0.375     , 0.        ],
       [0.04166667, 0.91666667],
       [0.        , 0.875     ],
       [0.45833333, 0.29166667],
       [0.625     , 0.41666667],
       [0.08333333, 0.625     ],
       [0.125     , 1.        ],
       [0.54166667, 0.29166667],
       [0.54166667, 0.5       ],
       [0.58333333, 0.45833333],
       [0.04166667, 0.70833333],
       [0.5       , 0.375     ],
       [0.91666667, 0.41666667],
       [0.04166667, 0.41666667],
       [0.91666667, 0.5       ],
       [0.5       , 0.125     ],
       [0.91666667, 0.5       ],
       [0.04166667, 0.66666667],
       [0.

In [31]:
# DataLoader lets us load data in batches, and can be used to train NN.
# Convert input_train into TENSORS
input_train_tensors = torch.tensor(input_train.values).type(torch.float32)
input_train_tensors[:5]

tensor([[0.7500, 0.3333],
        [0.7917, 0.3333],
        [0.3750, 0.1667],
        [0.5000, 0.3333],
        [0.5000, 0.2083]])

In [32]:
# Convert input_test into TENSORS
input_test_tensors = torch.tensor(input_test.values).type(torch.float32)
input_test_tensors[:5]

tensor([[0.0417, 0.5000],
        [0.6250, 0.5417],
        [0.5000, 0.3333],
        [0.5000, 0.1250],
        [0.0417, 0.4167]])

In [33]:
# Combine input_train_tensors with the ONE HOT ENCODED labels. Load into DataLoader our TensorDataset.
train_dataset = TensorDataset(input_train_tensors, one_hot_label_train)
train_dataloader = DataLoader(train_dataset)

In [34]:
train_dataset

In [35]:
train_dataloader

* Building a neural network with PyTorch means creating a new class. And to make it easy to train the neural network, this class will inherit from LightningModule.

* Our new class will have the following methods:

  - __init__() to initialize the Weights and Biases and keep track of a few other housekeeping things.

  - forward() to make a forward pass through the neural network.

  - configure_optimizers() to configure the optimizer. There are lots of optimizers to choose from, but in this tutorial, we'll change things up and use Adam.

  - training_step() to pass the training data to forward(), calculate the loss and keep track of the loss values in a log file.

In [36]:
class MultipleInsOuts(L.LightningModule):
  
  def __init__(self):
    super().__init__()
    
    L.seed_everything(seed=42)
    
    self.input_to_hidden = nn.Linear(in_features=2, out_features=2, bias=True)
    
    self.hidden_to_output = nn.Linear(in_features=2, out_features=3, bias=True)
    
    self.loss = nn.CrossEntropyLoss()
    
  
  def forward(self, input):
    hidden = self.input_to_hidden(input)
    
    output_values = self.hidden_to_output(torch.relu(hidden))
    return(output_values)
  
  
  def configure_optimizers(self):
    return Adam(self.parameters(), lr=0.001)
  

  def training_step(self, batch, batch_idx):
    inputs, labels = batch
    
    outputs = self.forward(inputs)
    
    loss = self.loss(outputs, labels)
    return loss

In [37]:
model = MultipleInsOuts()

In [38]:
trainer = L.Trainer(max_epochs=10)
trainer.fit(model, train_dataloaders=train_dataloader)

/Users/svenwu/Hustle/MachineLearning/venv/lib/python3.14/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
/Users/svenwu/Hustle/MachineLearning/venv/lib/python3.14/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/svenwu/Hustle/MachineLearning/venv/lib/python3.14/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consid

Epoch 9: 100%|██████████| 112/112 [00:00<00:00, 238.31it/s, v_num=0]


In [40]:
predictions = model(input_test_tensors)

In [41]:
predictions[0:4, ]

tensor([[ 0.9644, -0.1377,  0.0748],
        [ 0.0507,  0.5664,  0.8702],
        [ 0.1509,  0.4944,  0.6980],
        [ 0.1030,  0.5353,  0.6764]], grad_fn=<SliceBackward0>)

In [42]:
predicted_labels = torch.argmax(predictions, dim=1)
predicted_labels[0:4]

tensor([0, 2, 2, 2])

In [43]:
torch.sum(torch.eq(torch.tensor(label_test), predicted_labels)) / len(predicted_labels)

tensor(0.6579)

In [46]:
path_to_checkpoint = trainer.checkpoint_callback.best_model_path

In [47]:
# Create new Lightning Trainer
trainer = L.Trainer(max_epochs=100)

trainer.fit(model, train_dataloaders=train_dataloader, ckpt_path=path_to_checkpoint)

/Users/svenwu/Hustle/MachineLearning/venv/lib/python3.14/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:566: The dirpath has changed from '/Users/svenwu/Hustle/MachineLearning/neural_networks/statquest/test_runs/lightning_logs/version_0/checkpoints' to '/Users/svenwu/Hustle/MachineLearning/neural_networks/statquest/test_runs/lightning_logs/version_1/checkpoints', therefore `best_model_score`, `kth_best_model_path`, `kth_value`, `last_model_path` and `best_k_models` won't be reloaded. Only `best_model_path` will be reloaded.
/Users/svenwu/Hustle/MachineLearning/venv/lib/python3.14/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/svenwu/Hustle/MachineLearning/venv/lib/python3.14/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increas

Epoch 99: 100%|██████████| 112/112 [00:00<00:00, 181.72it/s, v_num=1]


In [48]:
predictions = model(input_test_tensors)
predicted_labels = torch.argmax(predictions, dim=1)

torch.sum(torch.eq(torch.tensor(label_test), predicted_labels)) / len(predicted_labels)

tensor(0.9474)

In [49]:
for name, param in model.named_parameters():
    print(name, torch.round(param.data, decimals=2))

input_to_hidden.weight tensor([[ 3.5500,  0.2500],
        [-1.9600,  1.4800]])
input_to_hidden.bias tensor([-0.3400,  1.2400])
hidden_to_output.weight tensor([[-4.2000,  3.1500],
        [ 0.1600,  0.3300],
        [ 1.8700, -3.2900]])
hidden_to_output.bias tensor([ 0.4600,  0.9600, -0.6100])


In [50]:
normalized_values = ([0.2, 3.0] - min_vals_in_input_train) / (max_vals_in_input_train - min_vals_in_input_train)
normalized_values

petal_width    0.041667
sepal_width    0.416667
dtype: float64

In [51]:
# torch.argmax(model(torch.tensor(normalized_values).type(torch.float32)))
input_tensor = torch.tensor(normalized_values.values, dtype=torch.float32)

torch.argmax(model(input_tensor))

tensor(0)